In [ ]:
# Example 1: validate well-formed RFC3339 timestamp strings
from strict_rfc3339 import validate_rfc3339

valid_examples = [
    "2023-01-15T12:34:56Z",
    "2023-01-15T12:34:56.789Z",
    "2023-01-15T12:34:56+00:00",
    "2023-01-15T12:34:56-05:00",
    "1985-04-12T23:20:50.52Z",
]

for ts in valid_examples:
    ok = validate_rfc3339(ts)
    print(f"  {ts!r:40s} -> {ok}")
    assert ok is True, f"expected {ts!r} to be valid"

print("\nAll well-formed timestamps validated successfully")


In [ ]:
# Example 2: reject malformed RFC3339 timestamps
from strict_rfc3339 import validate_rfc3339

invalid_examples = [
    "not-a-date",
    "2023-01-15 12:34:56",            # space instead of T
    "2023-01-15T12:34:56",            # missing tz
    "2023-13-15T12:34:56Z",           # month=13
    "2023-01-32T12:34:56Z",           # day=32
    "2023-01-15T25:34:56Z",           # hour=25
    "",
]

for ts in invalid_examples:
    ok = validate_rfc3339(ts)
    print(f"  {ts!r:40s} -> {ok}")
    assert ok is False, f"expected {ts!r} to be invalid"

print("\nAll malformed timestamps rejected as expected")


In [ ]:
# Example 3: convert RFC3339 string -> Unix timestamp (seconds since epoch)
from strict_rfc3339 import rfc3339_to_timestamp

# UTC epoch reference
epoch = rfc3339_to_timestamp("1970-01-01T00:00:00Z")
print(f"1970-01-01T00:00:00Z -> {epoch}")
assert epoch == 0, f"expected 0, got {epoch}"

# A known timestamp
ts = rfc3339_to_timestamp("2023-01-15T12:34:56Z")
print(f"2023-01-15T12:34:56Z -> {ts}")
assert isinstance(ts, (int, float))
assert ts > 0

# Equivalent forms across timezones must agree on absolute time
utc_form = rfc3339_to_timestamp("2023-01-15T12:00:00Z")
off_form = rfc3339_to_timestamp("2023-01-15T07:00:00-05:00")
print(f"UTC vs -05:00 equivalent: {utc_form} == {off_form}")
assert utc_form == off_form


In [ ]:
# Example 4: convert Unix timestamp -> RFC3339 string
from strict_rfc3339 import timestamp_to_rfc3339_utcoffset, timestamp_to_rfc3339_localoffset, validate_rfc3339

# Epoch round-trip
s = timestamp_to_rfc3339_utcoffset(0)
print(f"0 -> {s}")
assert validate_rfc3339(s) is True
assert s.startswith("1970-01-01T00:00:00")

# A non-zero timestamp
s2 = timestamp_to_rfc3339_utcoffset(1_700_000_000)
print(f"1_700_000_000 -> {s2}")
assert validate_rfc3339(s2) is True

# Local-offset variant should also produce a valid RFC3339 string
s3 = timestamp_to_rfc3339_localoffset(1_700_000_000)
print(f"local offset    -> {s3}")
assert validate_rfc3339(s3) is True


In [ ]:
# Example 5: full round-trip RFC3339 <-> timestamp + InvalidRFC3339Error on bad input
import strict_rfc3339
from strict_rfc3339 import (
    rfc3339_to_timestamp,
    timestamp_to_rfc3339_utcoffset,
    validate_rfc3339,
)

original = "2023-06-15T14:30:00Z"
ts = rfc3339_to_timestamp(original)
round_tripped = timestamp_to_rfc3339_utcoffset(ts)
print(f"{original} -> {ts} -> {round_tripped}")

assert validate_rfc3339(round_tripped) is True
assert rfc3339_to_timestamp(round_tripped) == ts

# Bad-input behavior: rfc3339_to_timestamp raises on invalid input
raised = False
try:
    rfc3339_to_timestamp("definitely-not-rfc3339")
except strict_rfc3339.InvalidRFC3339Error as e:
    raised = True
    print(f"raised {type(e).__name__}: {e}")
assert raised, "expected an exception for malformed input"

print("\nRound-trip + error path OK")
